# Chapter 9 - RF Cavities

## Goals

This chapter introduces reference energy and RF-cavity elements, then inserts RF cavities into
the 10 o'clock straight section of the ring. The numerical settings follow the original Bmad
tutorial. Parameter names and phase units are translated to SciBmad.

By the end of the chapter, we will:

1. distinguish the reference energy and reference momentum from particle phase-space coordinates;
2. compare a reference-energy-changing cavity section with an ordinary `RFCavity`;
3. construct the special `FODORF` cell;
4. replace four ordinary straight FODO cells with RF-cavity cells.

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()


In [5]:
using SciBmad
using Printf

tutorial_root = @__DIR__
include(joinpath(tutorial_root, "lattices", "chapter_5", "chapter5_ring_definition.jl"))

const C5 = Chapter5Ring

Main.Chapter5Ring

## 9.1 Reference Energy and the Cavity and RFCavity Elements

### Reference Energy

Every lattice element has a reference particle, a reference energy called `E_ref`, and a
reference momentum named `pc_ref` (in eV). The three are interrelated so that knowing the
reference particle and either the reference momentum or energy allows the third quantity to be
calculated. The reference momentum is used for normalization of the phase-space momentum as well
as normalized strength parameters.

For example, for a quadrupole element, the `Bn1` parameter (units: Tesla/m) can be used to
specify the linear field gradient and the `Kn1` parameter (units: 1/m$^2$) is the normalized
field gradient normalized using the value of the reference momentum.

### Cavity

A reference-energy-changing cavity is an exception for lattices with an open geometry, where the
reference energy/momentum at the beginning of the lattice is what is set in the lattice file.
Usually, the reference energy/momentum for a downstream element inherits the reference
energy/momentum of the element just upstream.

A reference-energy-changing cavity represents an RF cavity where the reference energy/momentum
at the exit end of the cavity is set so that a particle entering the cavity with zero phase-space
coordinates leaves with zero phase-space coordinates. In particular, its phase-space `pz` will
be zero at the exit end.

### RFCavity

`RFCavity` elements also represent an RF cavity, but their reference energy/momentum is the
same as the upstream lattice section. The energy gained or lost by the tracked particle therefore
appears in its longitudinal phase-space coordinate.

### Bmad-to-SciBmad parameter translation

| Original Bmad name | SciBmad name or representation |
|---|---|
| `E_tot` | `E_ref` |
| `p0c` | `pc_ref` |
| `k1` | `Kn1` |
| `b1_gradient` | `Bn1` |
| `rfcavity` | `RFCavity` |
| `lcavity` reference-energy change | a new lattice/beamline section using `dE_ref` |
| Bmad `phi0` in turns | SciBmad `phi0` in radians |

SciBmad currently exposes one cavity element constructor, `RFCavity`. A changing reference
energy is represented at the lattice-section level with `dE_ref`; it is not a separate
`LCavity` element type.

In [6]:
example_species = Species("positron")
example_pc_ref = 1e8

q1_reference = Quadrupole(name="Q1_REFERENCE", L=0.1, Kn1=0.14)
q2_reference = Quadrupole(
    name="Q2_REFERENCE",
    L=0.1,
    Bn1=example_pc_ref * q1_reference.Kn1 / C_LIGHT,
)
reference_line = Beamline(
    [q1_reference, q2_reference];
    species_ref=example_species,
    pc_ref=example_pc_ref,
)

@printf("Reference pc = %.6e eV\n", reference_line.pc_ref)
@printf("Reference total energy = %.6e eV\n", reference_line.E_ref)
@printf("Q1 normalized gradient Kn1 = %.6e 1/m^2\n", q1_reference.Kn1)
@printf("Q2 physical gradient Bn1 = %.6e T/m\n", q2_reference.Bn1)
@printf("Q2 normalized gradient Kn1 = %.6e 1/m^2\n", q2_reference.Kn1)

Reference pc = 1.000000e+08 eV
Reference total energy = 1.000013e+08 eV
Q1 normalized gradient Kn1 = 1.400000e-01 1/m^2
Q2 physical gradient Bn1 = 4.669897e-02 T/m
Q2 normalized gradient Kn1 = 1.400000e-01 1/m^2


## 9.2 Example: Cavity Element

The original example uses the following numerical settings:

- positron reference particle;
- beginning reference momentum `pc_ref = 1e8 eV`;
- two quadrupoles of length `0.1 m`;
- `Q1[Kn1] = 0.14 1/m^2`;
- a reference-energy-changing cavity of length `1 m`, voltage `1e9 V`, and frequency
  `1e9 Hz`;
- an ordinary RF cavity of length `1 m`, voltage `1e8 V`, and the same frequency.

For the ordinary RF cavity, the original Bmad phase `phi0 = 0.25` turns becomes
`phi0 = pi/2` radians in SciBmad. This is peak acceleration for the positron used here.

The first calculation below represents the reference-energy-changing cavity by starting a new
SciBmad lattice section with `dE_ref = 1e9`. The second calculation tracks a particle through
an ordinary `RFCavity`, whose reference momentum stays fixed.

In [7]:
q1_upstream = Quadrupole(
    name="Q1_UPSTREAM",
    L=0.1,
    Kn1=0.14,
    pc_ref=example_pc_ref,
    species_ref=example_species,
)
q2_upstream = Quadrupole(
    name="Q2_UPSTREAM",
    L=0.1,
    Bn1=example_pc_ref * 0.14 / C_LIGHT,
)

lc_like = RFCavity(
    name="LC_LIKE",
    L=1.0,
    voltage=1e9,
    rf_frequency=1e9,
    phi0=0.0,
    dE_ref=1e9,
    species_ref=example_species,
)

q1_downstream = Quadrupole(name="Q1_DOWNSTREAM", L=0.1, Kn1=0.14)
q2_downstream = Quadrupole(
    name="Q2_DOWNSTREAM",
    L=0.1,
    Bn1=example_pc_ref * 0.14 / C_LIGHT,
)
rf_downstream = RFCavity(
    name="RF",
    L=1.0,
    voltage=1e8,
    rf_frequency=1e9,
    phi0=pi/2,
)

cavity_lattice = Lattice([
    q1_upstream,
    q2_upstream,
    lc_like,
    q1_downstream,
    q2_downstream,
    rf_downstream,
])

for (i, branch) in enumerate(cavity_lattice.beamlines)
    @printf(
        "Section %d: E_ref = %.6e eV, pc_ref = %.6e eV, elements = %s\n",
        i,
        branch.E_ref,
        branch.pc_ref,
        join([element.name for element in branch.line], ", "),
    )
end

Section 1: E_ref = 1.000013e+08 eV, pc_ref = 1.000000e+08 eV, elements = Q1_UPSTREAM, Q2_UPSTREAM
Section 2: E_ref = 1.100001e+09 eV, pc_ref = 1.100001e+09 eV, elements = LC_LIKE, Q1_DOWNSTREAM, Q2_DOWNSTREAM, RF


In [8]:
rf_example = RFCavity(
    name="RF_TRACKING_EXAMPLE",
    L=1.0,
    voltage=1e8,
    rf_frequency=1e9,
    phi0=pi/2,
)
rf_example_line = Beamline(
    [rf_example];
    species_ref=example_species,
    pc_ref=example_pc_ref,
)
particle = Bunch(
    zeros(6);
    species=rf_example_line.species_ref,
    p_over_q_ref=rf_example_line.p_over_q_ref,
)

track!(particle, rf_example_line)

@printf("RF line reference pc remains %.6e eV\n", rf_example_line.pc_ref)
@printf("Tracked particle final pz = %.9f\n", particle.coords.v[1, 6])

RF line reference pc remains 1.000000e+08 eV
Tracked particle final pz = 1.000006529


## 9.3 Adding RF Cavities

In a real machine, electrons emit synchrotron radiation as they bend around the ring, so we need
to replenish the energy with an RF system. Let's put RF cavities in the 10 o'clock straight
section of our ring. We will design a special FODO cell with RF cavities, `FODORF`, and replace
four FODO cells in the 10 o'clock straight section.

In the ESR, there are two RF cavities of length $4.017\,\mathrm{m}$ between each quadrupole with
equal drift spaces between each element. We will follow these dimensions as shown below.

<div align="center"><img src="assets/chapter9_rf_fodo_cell.png" width="825"></div>

## 9.4 Example: Constructing the FODO Cell with RF Cavities

The construction follows the original example:

1. Define the RF cavities and equal drift elements.
2. Define the `FODORF` cell.
3. Replace four FODO cells in the 10 o'clock straight section by modifying the Chapter 5
   equivalents of `SEXTANT9` and `SEXTANT11`.

The RF settings are kept exactly as specified in the PDF:

$$
L_{RF}=4.017\ \mathrm{m},\qquad h=7560,\qquad
V=\frac{68}{18}\times10^6\ \mathrm{V}.
$$

The PDF expression for the equal drift gives a negative length when combined with the Chapter 5
straight-section dimensions. Instead, this SciBmad example uses a conventional RF-region spacing
of $0.3\,\mathrm{m}$ between elements while retaining the PDF cavity length:

$$
L_{DRF}=0.3\ \mathrm{m}.
$$

This is not a drift length repeatedly used by the preceding tutorial chapters: those chapters
mainly use `D1 = 0.609 m`, `D2 = 1.241 m`, and `DB = 5.855 m`. The $0.3\,\mathrm{m}$ choice is a
reasonable RF-region equipment spacing. Since the cavity length remains $4.017\,\mathrm{m}$, the
new `FODORF` cell is longer than the ordinary straight FODO cell and increases the ring circumference.

In [9]:
const RF_L = 4.017
const RF_HARMON = 7560
const RF_VOLTAGE = 68.0 / 18.0 * 1e6
const DRF_L = 0.3

@printf("RF cavity length = %.6f m\n", RF_L)
@printf("RF harmonic number = %d\n", RF_HARMON)
@printf("Starting RF voltage = %.6e V\n", RF_VOLTAGE)
@printf("RF-region drift length = %.9f m\n", DRF_L)

RF cavity length = 4.017000 m
RF harmonic number = 7560
Starting RF voltage = 3.777778e+06 V
RF-region drift length = 0.300000000 m


In [10]:
function make_FODORF_elements(; voltage=RF_VOLTAGE)
    rf() = RFCavity(
        name="RF0",
        L=RF_L,
        harmon=RF_HARMON,
        voltage=voltage,
    )
    drf() = Drift(name="DRF", L=DRF_L)

    return [
        C5.make_element(:QFSS),
        drf(), rf(), drf(), rf(), drf(),
        C5.make_element(:QDSS),
        drf(), rf(), drf(), rf(), drf(),
    ]
end

FODORF = make_FODORF_elements()

ordinary_fodo_length = sum(element.L for element in C5.make_elements(C5.FODOSSF))
rf_fodo_length = sum(element.L for element in FODORF)

@printf("Ordinary straight FODO length = %.9f m\n", ordinary_fodo_length)
@printf("FODORF length = %.9f m\n", rf_fodo_length)
@printf("Length increase per FODORF cell = %.9f m\n", rf_fodo_length - ordinary_fodo_length)
println("FODORF element order:")
println(join([element.name for element in FODORF], " -> "))

Ordinary straight FODO length = 16.410000000 m
FODORF length = 18.868000000 m
Length increase per FODORF cell = 2.458000000 m
FODORF element order:
QFSS -> DRF -> RF0 -> DRF -> RF0 -> DRF -> QDSS -> DRF -> RF0 -> DRF -> RF0 -> DRF


In [11]:
repeat_FODORF(n; voltage=RF_VOLTAGE) =
    reduce(vcat, (make_FODORF_elements(voltage=voltage) for _ in 1:n))

function build_ring_with_rf(; voltage=RF_VOLTAGE)
    sextant9_rf = vcat(
        C5.make_elements(C5.repeat_line(C5.FODOSSF, 4)),
        C5.make_elements(C5.SS_TO_ARCF),
        C5.make_elements(C5.repeat_line(C5.FODOAF, 20)),
        C5.make_elements(C5.ARC_TO_SSF),
        C5.make_elements(C5.repeat_line(C5.FODOSSF, 2)),
        repeat_FODORF(2; voltage=voltage),
    )

    sextant11_rf = vcat(
        repeat_FODORF(2; voltage=voltage),
        C5.make_elements(C5.repeat_line(C5.FODOSSR, 2)),
        C5.make_elements(C5.SS_TO_ARCR),
        C5.make_elements(C5.repeat_line(C5.FODOAR, 20)),
        C5.make_elements(C5.ARC_TO_SSR),
        C5.make_elements(C5.repeat_line(C5.FODOSSR, 4)),
    )

    elements = vcat(
        C5.make_elements(C5.SEXTANT1),
        C5.make_elements(C5.SEXTANT3),
        C5.make_elements(C5.SEXTANT5),
        C5.make_elements(C5.SEXTANT7),
        sextant9_rf,
        sextant11_rf,
    )
    ring = Beamline(elements; species_ref=C5.species_ref, E_ref=C5.E_ref)
    return ring, elements
end

ring_with_rf, ring_with_rf_elements = build_ring_with_rf()

@printf("Complete ring elements: %d\n", length(ring_with_rf_elements))
@printf("Installed RF cavities: %d\n",
        count(element -> element.name == "RF0", ring_with_rf_elements))
@printf("Complete ring circumference: %.9f m\n",
        sum(element.L for element in ring_with_rf_elements))
@printf("Circumference increase from four FODORF cells: %.9f m\n",
        4 * (rf_fodo_length - ordinary_fodo_length))

Complete ring elements: 1744
Installed RF cavities: 16
Complete ring circumference: 3843.832000000 m
Circumference increase from four FODORF cells: 9.832000000 m


In [12]:
rf_indices = findall(element -> element.name == "RF0", ring_with_rf_elements)
first_rf = ring_with_rf_elements[first(rf_indices)]

@printf("First RF0: L = %.6f m\n", first_rf.L)
@printf("First RF0: harmonic = %.0f\n", first_rf.harmon)
@printf("First RF0: voltage = %.6e V\n", first_rf.voltage)
@printf("RF cavities occupy element indices %d through %d\n",
        first(rf_indices), last(rf_indices))

First RF0: L = 4.017000 m
First RF0: harmonic = 7560
First RF0: voltage = 3.777778e+06 V
RF cavities occupy element indices 1427 through 1471


### What this example does not do

This example constructs and installs the RF-cavity cells. It does not optimize the transverse
optics or RF voltage. Setting a target synchrotron tune is left as exercise.

## 9.5 Exercises

1. **Set the synchrotron tune.** Vary the RF cavity voltages to set the synchrotron tune to
   $0.05$, and write down the new `RF0.voltage`.
2. Modify the reference-energy-changing cavity example to have a small length, and set the
   beginning momentum small enough that the relativistic beta is significantly less than one.
   Starting the particle with a finite $z$, calculate the ending $z$ and verify the result with
   SciBmad.
3. Vary the RF phase seen by the particle without changing the reference energy. In SciBmad,
   apply the offset through `phi0` in radians and determine approximately which phase range
   decelerates the particle enough that it turns around before passing through the cavity.